# Knee MRI model evaluation

When two knee MRI models both score 0.958, what can that number tell us about how they will perform on new examinations? This notebook explores that question with historical score summaries and a controlled simulation.

The underlying imaging task uses knee MRI examinations paired with their radiology reports. A detector would return twelve confidence scores per examination, one for each target finding. Examples include ligament tears, meniscal damage, cartilage loss, and fractures. Here the unit of analysis is model performance. All study-level labels and predictions used below are generated in the notebook.

**How to read the metric.** For one finding, ROC AUC measures how often a randomly chosen positive study receives a higher confidence score than a negative study. A tie counts as half. Macro AUC is the arithmetic mean of the twelve per-finding AUCs, so each finding has equal weight. A score of 0.958 describes how well the model orders studies across findings. Measuring diagnostic accuracy would also require choosing a threshold for each finding.

## What a rounded validation score can hide

The original evaluation recorded two score summaries on October 23, 2026. Its author matched the summaries by source ID and removed the identifiers. Each matched record contains a validation macro AUC and the best holdout macro AUC from up to two selected runs. The two values can therefore describe different runs. The available data contain scores rounded to three decimals.

The data cell stores each distinct pair once, with the number of times it occurs. The first two values in each group of three are validation and holdout AUC in thousandths. The third is the record count.

In [1]:
import numpy as np

# Groups of three integers encode validation AUC, holdout AUC (both in thousandths), and count.
score_pair_table = """
970 957 1   969 960 2   969 958 2   968 959 1   968 957 1   968 956 2   968 955 1   967 958 1
967 957 2   967 956 1   967 955 1   967 953 1   966 956 1   966 954 2   966 952 2   965 956 1
965 955 3   965 954 1   965 953 1   965 950 2   965 949 1   964 955 1   964 954 2   964 952 4
964 951 4   964 950 1   963 953 2   963 952 1   963 951 2   963 950 3   963 949 1   963 948 2
963 947 2   962 952 3   962 951 3   962 950 2   962 949 7   962 948 4   962 947 1   962 946 2
961 953 1   961 950 2   961 949 2   961 948 7   961 947 7   961 946 8   961 945 2   961 944 1
960 949 4   960 948 9   960 947 7   960 946 12   960 945 7   960 944 9   960 943 3   959 948 3
959 947 8   959 946 13   959 945 24   959 944 33   959 943 15   959 942 2   958 946 3   958 945 12
958 944 53   958 943 272   958 942 1239   958 941 24   958 940 7   958 939 3   958 938 2   958 937 2
958 936 1   958 935 1   958 934 1   958 933 2   958 931 1   958 930 1   958 927 4   958 926 1
958 925 1   958 923 3   958 922 1   958 920 1   958 919 1   958 909 1   958 899 1   957 945 2
957 944 2   957 943 10   957 942 66   957 941 82   957 940 2   957 939 2   957 938 2   957 937 2
957 932 2   957 929 1   957 926 1   956 945 1   956 943 3   956 942 9   956 941 108   956 940 95
956 939 1   956 938 3   956 934 1   956 932 2   956 931 1   956 930 1   956 929 1   956 928 2
956 924 1   956 923 1   956 922 1   956 909 1   955 944 2   955 942 4   955 941 3   955 940 2
955 939 7   955 938 8   955 935 1   954 942 2   954 941 1   954 940 2   954 939 10   954 938 32
954 937 52   954 935 1   954 934 1   954 933 1   954 930 1   954 929 1   954 927 3   954 923 1
954 920 1   954 898 1   953 941 1   953 940 1   953 939 4   953 938 2   953 937 5   953 936 2
953 935 1   953 931 1   952 939 1   952 938 2   952 937 1   952 935 3   951 943 1   951 941 1
951 939 2   951 938 3   951 937 6   951 936 3   951 935 2   951 934 2   951 933 1   951 932 2
951 931 1   951 928 1   951 921 1   951 911 1   950 938 1   950 937 1   950 936 4   950 935 10
950 934 2   950 933 18   950 932 66   950 931 14   950 929 1   950 927 1   950 926 1   950 921 1
950 912 1   950 848 1   949 938 1   949 937 1   949 935 3   949 934 1   949 933 5   949 932 12
949 931 3   949 930 25   949 929 1   949 927 1   949 922 1   949 921 1   949 862 1   949 765 1
949 499 1   948 938 2   948 936 1   948 935 2   948 934 2   948 933 2   948 932 4   948 931 1
948 930 1   948 929 2   947 937 1   947 934 1   947 933 3   947 932 1   947 931 1   947 930 1
947 928 2   946 934 1   946 933 1   946 932 3   946 931 2   946 930 2   946 929 1   946 928 1
946 926 2   945 933 1   945 931 5   945 930 10   945 929 8   945 928 2   945 926 3   944 931 2
944 930 6   944 929 37   944 928 7   944 927 3   944 926 1   944 915 1   943 931 2   943 930 1
943 929 11   943 928 17   943 927 269   943 926 4   943 925 3   943 924 1   943 923 3   943 922 1
943 921 2   943 916 1   943 903 1   943 864 1   943 839 1   943 817 1   943 733 1   942 930 1
942 929 2   942 928 4   942 927 9   942 926 20   942 925 63   942 924 1   942 914 1   942 693 1
941 928 2   941 927 5   941 926 7   941 925 12   941 924 11   941 923 141   941 922 1   941 921 1
941 919 1   940 933 1   940 929 1   940 928 1   940 927 2   940 926 4   940 925 2   940 924 4
940 923 41   940 922 45   940 921 2   940 920 2   940 919 1   940 913 1   940 872 1   940 800 1
939 928 1   939 927 2   939 926 3   939 924 5   939 923 2   939 922 4   939 921 28   939 920 2
939 890 1   938 928 1   938 927 1   938 926 2   938 924 6   938 923 7   938 922 1   938 921 2
938 920 1   938 919 1   937 928 1   937 925 1   937 924 4   937 923 1   937 922 3   937 921 4
937 920 10   937 919 17   937 917 1   936 926 1   936 925 2   936 924 1   936 923 1   936 922 2
936 921 13   936 920 74   936 919 104   936 917 1   936 916 1   936 915 1   936 899 1   936 868 1
936 817 1   936 500 1   935 923 2   935 922 2   935 921 4   935 920 3   935 919 24   935 918 3
935 917 2   935 914 1   935 818 1   934 923 1   934 921 2   934 919 3   934 918 3   934 917 3
934 916 2   933 923 1   933 921 1   933 919 2   933 918 4   933 917 5   933 916 1   933 915 2
933 914 1   933 912 2   933 911 1   932 919 4   932 918 3   932 917 4   932 916 10   931 919 1
931 918 2   931 915 2   931 914 2   930 919 1   930 917 4   930 916 1   930 915 2   930 913 5
930 912 1   930 911 1   930 910 1   930 908 1   929 918 2   929 917 1   929 916 7   929 915 2
929 914 3   929 912 5   929 910 2   928 916 2   928 915 11   928 914 2   928 913 2   928 908 2
928 907 1   927 916 2   927 915 3   927 914 5   927 913 2   927 912 2   927 911 1   927 910 1
927 909 1   927 908 1   927 905 1   926 915 1   926 913 2   926 912 2   926 911 3   926 910 2
926 909 1   926 908 1   926 903 1   926 902 1   926 500 1   925 914 2   925 913 4   925 912 1
925 910 1   925 908 3   925 897 1   924 917 1   924 913 1   924 912 34   924 911 1   924 909 2
924 907 1   924 906 1   924 905 1   924 904 1   924 903 1   924 882 1   923 913 1   923 910 4
923 909 2   923 907 2   923 906 1   923 904 1   923 902 1   923 900 1   922 912 1   922 910 1
922 909 1   922 908 1   922 905 1   922 904 1   922 901 2   922 900 6   922 899 21   922 876 1
921 907 1   921 906 4   921 905 1   921 904 1   921 903 1   921 902 1   921 901 1   921 900 8
921 899 11   920 910 2   920 906 2   920 905 3   920 899 6   920 898 29   920 897 1   919 909 1
919 908 1   919 907 1   919 905 1   919 904 1   919 903 1   919 901 3   919 898 1   919 798 1
918 906 1   918 905 1   918 904 1   918 903 3   918 901 1   918 898 1   918 897 1   918 896 1
917 907 1   917 905 3   917 902 2   917 901 2   917 897 1   917 896 4   917 895 1   917 894 1
917 836 1   916 909 1   916 908 2   916 903 2   916 902 1   916 900 2   916 898 1   916 897 1
916 896 1   916 895 1   915 906 2   915 903 1   915 902 1   915 898 1   915 897 1   915 896 1
915 895 7   915 894 1   914 903 1   914 901 2   914 900 2   914 899 1   914 898 1   914 897 2
914 896 1   914 894 18   913 903 1   913 901 1   913 900 2   913 899 1   913 896 2   913 894 1
913 893 2   913 890 1   912 905 1   912 899 1   912 897 1   912 895 1   912 894 1   912 893 1
912 891 1   911 897 2   911 896 2   911 894 2   911 893 2   911 892 1   911 891 7   911 890 19
911 871 1   910 902 2   910 899 1   910 898 1   910 895 1   910 894 1   910 893 1   910 892 1
910 891 3   910 890 1   910 889 7   910 887 1   910 884 1   909 899 1   909 898 1   909 897 1
909 896 1   909 895 1   909 893 2   909 892 2   909 891 4   909 890 2   909 889 13   909 888 1
909 887 1   909 886 1   908 899 1   908 896 1   908 893 1   908 887 2   908 886 1   907 891 1
907 890 1   907 889 1   907 887 2   907 886 2   906 898 1   906 894 2   906 890 2   905 898 1
905 893 1   905 892 1   905 890 2   905 886 3   905 885 1   905 884 1   905 883 1   904 897 1
904 893 1   904 892 1   904 890 1   904 888 1   904 885 1   904 884 1   904 881 1   903 891 2
903 889 1   903 888 1   903 887 1   903 885 3   903 883 1   903 882 1   903 880 1   903 879 4
903 732 1   902 892 1   902 889 1   902 888 2   902 886 1   902 882 1   902 880 1   902 879 1
902 875 1   901 891 1   901 889 2   901 888 1   901 885 1   901 884 1   901 880 1   901 879 2
901 878 1   900 886 1   900 881 1   900 879 1   900 878 1   900 875 2   900 873 1   900 870 1
899 885 2   899 881 1   899 880 1   899 879 7   899 878 63   899 877 1   899 799 1   898 886 1
898 885 2   898 884 1   898 881 2   898 880 1   898 878 1   898 874 1   897 895 1   897 889 1
897 888 1   897 879 2   897 878 1   897 877 1   897 876 2   897 874 11   896 886 1   896 884 1
896 883 1   896 882 1   896 878 1   896 875 3   896 874 3   895 888 1   895 886 1   895 883 1
895 878 2   895 877 1   895 875 1   894 886 1   894 885 1   894 883 1   894 879 1   894 878 1
894 872 1   893 885 1   893 883 1   893 877 1   893 875 1   893 872 8   892 880 1   892 877 2
892 872 2   892 871 4   891 882 1   891 880 1   891 879 1   891 878 1   891 874 1   891 872 1
891 871 134   890 881 1   890 876 3   890 868 1   889 870 1   889 869 1   888 872 1   888 867 1
887 875 1   887 874 2   887 872 1   887 870 2   887 864 4   886 884 1   886 874 1   886 873 1
886 871 1   886 870 1   886 868 1   886 865 1   885 876 1   885 875 2   885 874 1   885 871 1
885 868 1   885 865 1   885 864 2   885 862 1   884 874 1   884 873 1   884 868 1   884 866 1
884 864 1   883 877 1   883 868 1   883 867 2   883 865 1   883 864 1   883 860 1   882 869 1
882 868 2   882 867 2   882 866 1   882 862 2   882 861 1   882 856 1   882 779 1   881 874 1
881 870 1   881 867 1   881 865 1   881 864 1   881 858 1   880 867 1   880 865 2   880 861 1
880 860 1   880 858 1   879 869 1   879 867 1   879 865 1   879 864 3   879 863 1   879 856 2
879 853 1   878 864 1   878 863 1   878 862 1   878 859 1   878 857 1   877 867 1   877 864 1
877 859 1   877 858 1   877 857 1   877 856 1   876 869 2   876 864 1   876 863 1   876 859 1
876 856 1   875 871 1   875 860 1   875 859 1   874 864 2   874 861 1   874 856 1   874 855 3
873 865 2   873 862 2   873 855 1   872 860 1   872 857 1   872 856 1   872 854 1   872 853 2
871 867 1   871 863 2   871 857 1   871 854 2   871 852 1   871 851 1   871 850 1   871 849 1
871 843 1   870 863 1   870 853 1   869 854 1   868 854 1   868 853 1   868 851 1   868 846 1
868 843 1   868 841 1   867 855 1   867 854 1   867 853 1   867 851 1   866 856 2   866 854 1
866 850 1   866 848 1   865 854 1   865 851 1   865 848 1   864 851 1   864 850 1   864 849 1
864 848 1   864 843 1   863 840 1   862 845 1   862 842 1   861 849 1   861 847 2   860 848 1
860 845 1   860 844 1   860 828 1   859 852 1   859 850 1   859 849 1   859 848 1   859 841 1
858 840 1   857 851 1   857 847 1   857 845 1   857 843 1   857 842 1   857 841 1   856 849 1
856 840 2   856 837 1   856 836 1   855 842 1   855 841 1   855 837 1   855 833 1   855 832 1
854 847 1   854 841 1   854 838 1   854 834 1   853 843 1   853 840 1   853 839 1   853 838 1
853 837 1   853 836 1   853 835 1   853 834 1   853 832 1   852 856 1   852 844 2   852 834 1
852 833 1   852 832 2   852 823 1   851 845 1   851 840 1   851 838 1   851 834 1   850 856 1
850 840 2   850 838 1   850 836 1   850 835 1   850 834 1   850 833 1   850 829 1   850 824 1
850 820 1   849 837 1   849 831 1   849 828 1   848 837 1   848 835 1   848 832 1   848 830 1
848 829 1   848 826 1   848 824 1   847 835 1   847 834 1   847 830 1   847 829 1   847 827 1
847 816 1   846 845 1   846 833 1   846 831 1   846 829 1   846 823 1   846 819 1   845 828 1
844 839 2   844 827 1   844 821 1   844 818 1   844 817 1   843 833 1   843 832 2   843 828 2
843 824 1   843 823 1   843 822 2   843 816 1   842 840 1   842 830 1   842 829 1   842 815 1
841 838 1   841 833 1   841 829 1   841 822 1   841 821 1   840 820 1   840 814 1   839 827 2
839 825 1   839 822 1   839 818 1   839 817 1   839 815 2   838 827 1   838 825 1   838 822 1
838 818 1   838 816 1   838 811 1   837 814 1   837 811 1   837 805 1   836 825 1   836 822 1
836 818 1   836 816 1   836 808 1   836 804 1   836 798 1   835 829 1   835 817 1   835 812 2
835 802 1   834 815 1   834 808 1   833 821 1   833 818 1   833 816 1   833 815 1   833 813 1
833 803 1   832 823 2   832 822 1   832 806 1   832 804 1   832 799 5   832 792 1   831 807 1
830 820 1   830 811 1   829 816 1   829 811 1   829 798 1   828 821 1   828 817 2   828 815 1
828 810 1   828 808 1   828 807 1   828 805 1   828 797 1   827 823 1   827 810 1   827 809 1
827 800 1   826 820 1   826 818 2   826 814 1   826 806 2   826 801 1   825 819 1   825 815 1
825 814 1   825 809 1   825 808 1   825 803 2   825 800 1   825 799 1   825 793 1   825 790 1
824 810 1   824 809 1   824 805 1   823 818 1   823 817 1   823 814 1   823 813 1   823 809 2
823 803 1   823 799 1   822 824 1   822 811 1   822 809 1   822 808 1   822 807 1   822 801 1
822 800 2   821 816 1   821 810 1   821 808 1   821 804 1   821 790 1   820 813 1   820 803 1
820 793 1   820 790 1   820 789 1   820 787 1   819 809 1   819 807 1   819 800 1   819 798 1
818 805 1   818 796 2   817 806 1   817 788 1   816 803 1   816 799 1   816 798 2   816 791 1
816 782 1   815 808 1   815 804 1   815 800 1   814 806 1   814 797 1   814 796 1   814 786 1
813 805 1   813 799 1   813 792 1   813 790 1   813 782 1   812 799 1   812 787 1   812 780 1
811 805 1   811 804 1   811 800 1   811 798 1   811 796 1   811 793 1   811 792 1   810 790 1
810 789 1   810 782 1   810 778 2   809 796 2   809 795 1   809 793 1   809 792 1   809 789 1
809 785 1   809 779 1   809 777 2   808 803 1   808 798 1   808 789 1   808 784 1   808 783 1
808 779 1   808 777 4   808 776 1   808 772 1   807 790 1   807 786 1   807 778 1   807 777 2
807 776 1   806 791 1   806 783 1   806 778 1   806 775 1   805 790 1   805 786 1   805 785 1
805 782 1   805 778 1   805 777 2   804 805 1   804 791 1   804 786 1   804 780 2   804 775 1
804 770 1   803 795 1   803 781 1   802 788 1   802 786 1   802 785 1   802 769 1   801 795 1
801 788 1   801 785 1   801 784 1   801 783 1   801 778 1   801 776 1   801 767 1   799 786 1
799 780 1   799 777 1   799 774 1   799 770 2   798 766 1   797 788 1   797 784 1   797 782 1
797 779 1   796 783 1   796 780 1   796 779 1   796 768 1   796 766 1   795 787 2   795 778 1
795 765 1   794 785 1   794 775 1   794 769 1   794 767 1   793 789 1   793 772 1   793 766 1
793 764 1   793 760 1   793 755 1   792 782 1   792 769 1   791 776 1   791 774 1   791 773 1
791 772 1   791 759 1   791 750 1   790 783 1   790 765 1   789 789 1   789 780 1   789 775 1
789 772 1   789 762 1   789 757 1   789 754 2   789 753 1   788 774 1   788 771 1   788 762 1
788 760 1   788 756 1   787 775 1   787 764 1   786 776 1   786 765 1   786 760 1   786 748 1
785 758 1   784 782 1   784 767 1   784 766 1   784 765 1   784 764 1   784 755 1   784 749 1
784 745 1   783 772 1   783 768 1   783 760 1   783 758 1   782 776 2   782 767 1   782 746 1
781 780 1   781 776 1   781 755 2   780 756 1   780 754 1   780 752 1   780 749 1   779 751 1
779 749 1   778 772 1   778 767 1   778 766 1   778 764 1   778 763 1   778 757 1   778 756 1
777 764 1   777 762 1   776 757 1   776 754 1   776 752 2   775 765 1   775 763 1   775 762 1
775 750 1   774 761 1   774 759 1   774 758 1   774 754 1   774 752 1   774 749 1   773 760 1
773 752 1   773 743 1   773 742 1   772 756 1   772 750 1   772 746 1   772 745 1   772 739 1
772 738 1   771 764 1   771 754 1   771 736 1   771 735 1   770 757 1   770 754 1   770 745 1
770 740 1   770 737 1   770 730 1   769 737 1   769 735 1   768 757 1   768 740 1   767 745 1
767 740 1   767 731 1   767 727 1   766 748 1   766 738 1   765 756 1   765 748 2   765 743 1
765 739 1   765 738 1   765 736 1   765 733 1   764 760 1   764 755 1   764 740 1   764 737 1
763 750 1   763 738 1   763 736 1   763 734 1   763 733 1   763 726 1   762 731 1   760 745 1
760 732 1   760 722 1   759 742 1   758 737 2   758 735 2   757 733 1   757 728 1   757 726 1
757 724 1   756 736 1   756 730 1   755 755 1   755 725 2   754 739 1   754 725 1   754 724 1
754 720 1   753 755 1   753 710 1   752 705 1   751 733 1   751 730 2   750 742 1   750 735 1
750 707 2   749 734 1   749 730 1   748 740 1   748 721 1   748 720 1   748 714 1   748 697 1
747 728 1   746 723 1   746 722 1   745 707 1   744 717 1   744 708 1   744 701 1   744 696 1
743 735 1   743 720 1   743 715 1   742 731 1   742 710 1   742 697 1   741 712 1   741 711 1
740 731 1   740 725 1   740 717 1   740 699 1   739 709 1   739 703 2   739 697 1   738 732 1
738 718 1   738 716 1   738 713 1   737 733 1   737 719 1   737 716 1   737 715 1   737 713 1
737 699 1   737 696 1   736 729 1   736 702 1   735 705 1   735 700 1   735 695 1   734 731 1
733 730 1   733 717 1   733 697 1   733 694 1   732 705 1   732 692 1   731 714 1   731 693 1
730 715 1   730 703 2   730 684 1   729 732 1   729 712 1   729 694 1   729 693 1   729 686 1
728 720 1   728 713 1   728 699 1   728 682 1   728 678 1   727 725 1   727 719 1   727 713 1
727 708 1   727 695 1   727 687 1   726 722 1   726 690 1   725 680 1   723 718 1   722 706 1
721 697 1   721 682 1   720 718 1   720 708 1   720 669 1   718 705 1   717 695 1   716 694 1
715 694 1   714 693 1   714 688 1   714 669 1   713 701 1   712 701 1   712 688 1   712 674 1
712 662 1   711 708 1   711 677 1   710 670 1   710 669 1   709 696 1   709 674 1   709 656 1
708 690 1   708 677 1   707 684 1   707 676 1   706 681 1   705 703 1   705 696 1   705 682 1
705 680 1   705 672 1   703 681 1   703 677 1   703 675 1   702 690 1   702 677 1   702 673 1
702 666 1   701 675 1   700 666 1   700 647 1   699 682 1   699 679 1   699 670 1   699 662 1
699 658 1   697 692 1   697 676 1   697 654 1   696 670 1   695 677 2   695 657 1   695 655 1
694 665 1   694 651 1   694 648 1   693 684 1   693 659 1   692 658 1   692 644 1   691 692 1
691 677 2   691 647 1   690 678 1   690 656 1   690 643 1   689 662 1   689 658 1   688 673 1
688 670 1   688 663 1   688 628 1   687 687 1   687 670 1   687 668 1   687 666 1   687 662 1
687 661 1   687 657 1   687 646 1   687 644 1   686 654 1   686 649 1   686 640 1   685 664 1
685 659 1   685 657 1   684 657 1   684 649 1   683 668 1   683 657 1   683 655 1   682 648 1
680 663 1   680 658 1   680 637 1   679 670 1   679 665 1   679 658 1   679 651 1   679 644 1
679 638 1   679 634 1   678 648 1   678 646 1   677 660 1   677 646 1   677 645 1   676 647 1
676 644 1   676 639 1   676 638 1   675 671 1   675 649 1   674 679 1   674 669 1   674 644 1
674 643 1   674 640 1   673 667 1   673 665 1   673 657 1   673 640 1   673 633 1   673 613 1
672 682 1   672 667 1   672 655 1   672 647 1   672 633 1   671 669 1   671 654 1   671 649 1
671 645 1   671 640 1   670 651 1   670 640 1   670 634 1   670 628 1   670 622 1   669 684 1
669 656 1   669 642 1   669 617 1   668 670 1   668 627 1   667 645 1   666 642 1   666 641 1
666 639 1   666 638 1   665 654 1   665 650 1   665 644 1   665 642 1   665 626 1   665 611 1
664 659 1   663 671 1   663 623 1   662 644 1   662 636 1   662 619 1   661 644 1   661 640 1
661 636 1   661 627 1   661 617 1   661 608 1   660 643 1   660 636 1   659 656 1   659 637 1
659 635 1   659 623 1   659 614 1   658 642 1   657 640 1   656 618 1   655 664 1   655 643 1
655 642 1   655 630 1   654 659 1   654 623 1   653 639 1   653 637 1   653 635 1   652 622 1
652 618 1   651 639 2   651 632 1   651 631 1   651 623 1   651 598 1   650 648 1   650 631 1
650 626 1   650 612 1   649 629 1   649 625 1   647 642 1   647 637 1   647 636 1   647 627 1
647 612 1   647 610 1   647 595 1   646 628 1   646 603 1   645 635 1   645 623 1   645 622 1
645 609 1   644 647 1   644 633 1   644 614 1   643 644 1   643 632 1   643 628 1   643 604 1
642 626 1   642 618 1   642 614 1   642 609 1   641 641 1   641 620 1   641 588 1   640 640 1
640 639 1   640 634 1   640 626 1   640 608 1   639 578 1   638 637 1   638 628 1   638 618 1
638 617 1   637 623 1   637 619 1   637 598 1   636 625 1   635 636 1   635 626 1   635 621 1
635 613 1   635 609 1   635 605 1   635 584 1   635 577 1   634 624 1   634 615 1   633 612 1
633 590 1   632 610 1   632 603 1   631 620 1   631 603 1   630 617 1   630 604 1   630 595 2
629 602 1   628 624 1   627 628 1   627 623 1   627 601 1   627 599 1   627 592 1   626 602 1
626 598 1   626 593 1   625 601 1   625 566 1   624 605 1   624 601 1   624 591 1   623 605 1
623 602 1   623 590 1   622 589 1   620 633 1   619 595 1   619 588 1   619 566 1   618 613 1
618 610 1   618 578 1   617 613 1   617 581 1   617 567 1   616 574 1   616 557 1   615 611 1
615 590 1   615 579 2   614 604 1   614 595 1   613 600 1   613 593 1   613 583 1   613 562 1
612 587 1   612 570 1   611 586 1   610 616 1   610 575 1   609 616 1   609 585 1   609 577 1
607 606 1   607 585 1   607 567 1   606 600 1   606 599 1   606 572 1   605 592 1   605 567 1
604 609 1   603 581 1   603 571 1   603 562 1   602 604 1   602 598 1   602 589 1   602 583 1
602 582 1   602 562 1   600 606 1   600 589 1   600 574 1   600 556 1   599 585 1   599 580 2
598 580 1   598 577 1   597 601 1   597 580 1   596 589 1   596 563 1   595 570 1   595 566 1
594 597 1   594 579 1   594 576 1   594 574 1   594 573 2   594 565 1   594 559 1   593 595 1
593 570 1   593 569 1   593 568 1   592 587 1   592 575 1   592 553 1   591 589 1   591 568 1
591 567 1   591 566 1   590 577 1   590 573 1   590 569 2   590 556 1   589 580 1   589 565 1
589 564 1   589 553 1   588 584 1   588 571 1   587 569 1   587 568 1   587 562 1   586 572 1
586 569 1   586 563 1   586 561 1   585 594 1   585 588 1   585 568 1   585 563 1   584 561 1
584 559 1   582 566 1   581 562 1   580 561 1   580 546 1   579 575 1   579 568 1   578 558 1
578 554 1   578 550 1   578 546 1   577 574 1   576 602 1   576 573 1   576 561 2   576 559 1
576 556 1   576 546 1   576 533 1   575 568 1   575 560 1   575 559 1   575 552 1   575 547 1
574 558 1   573 540 1   573 533 1   572 571 1   571 562 1   571 537 1   570 573 1   570 556 1
569 565 1   569 562 1   569 557 1   569 553 1   568 538 1   567 560 1   567 549 1   566 550 1
566 535 1   566 528 1   565 566 1   565 558 1   565 551 1   565 549 1   564 567 1   564 554 1
564 552 1   564 545 1   564 532 1   563 572 1   563 562 1   562 587 1   562 563 1   562 555 1
562 549 1   562 545 1   562 544 1   562 533 1   561 557 1   561 546 1   561 545 1   561 505 1
560 578 1   560 565 1   560 521 1   559 556 1   559 549 1   559 544 1   558 553 1   558 544 1
558 514 1   558 510 1   557 532 1   557 530 1   556 548 1   556 545 1   556 542 1   556 537 1
556 534 1   554 551 1   553 530 1   552 542 1   552 527 1   551 530 1   551 520 1   550 548 1
550 546 1   550 540 1   550 535 1   550 531 1   550 515 1   549 528 1   549 527 1   548 543 1
548 541 1   548 538 1   548 527 1   547 549 1   547 545 1   547 535 1   547 516 1   546 544 1
546 541 1   546 528 1   546 524 1   546 521 1   545 543 1   545 542 1   545 539 1   545 521 1
544 563 1   544 556 1   544 549 1   544 533 1   543 549 1   543 541 1   543 535 2   542 530 1
542 528 2   542 506 1   541 525 1   541 520 1   540 544 1   540 540 1   540 538 1   540 528 1
540 506 1   539 532 1   538 540 1   538 533 1   538 531 1   538 529 1   538 519 1   538 514 1
537 540 1   536 533 1   536 531 1   536 516 1   535 529 1   535 518 1   534 521 1   534 520 1
534 518 1   533 546 1   533 539 1   533 513 1   533 507 1   533 504 1   533 501 1   532 528 1
532 517 1   532 516 1   531 526 1   531 516 1   531 500 1   530 542 1   530 531 1   529 527 1
529 524 1   528 546 1   528 527 1   528 520 1   528 519 2   527 530 1   527 527 1   527 524 1
527 519 1   527 513 1   527 509 1   526 538 1   526 530 1   526 526 1   526 520 1   526 517 1
526 508 1   525 531 1   525 526 1   525 518 1   524 524 1   524 515 1   523 539 1   523 516 1
523 510 1   523 501 1   522 520 1   522 518 1   521 530 1   520 527 1   520 512 1   520 509 1
519 520 1   519 502 1   518 529 1   518 518 1   518 515 1   518 514 1   518 512 1   518 506 1
518 504 2   518 499 2   517 517 1   517 507 1   516 515 1   516 512 1   516 501 1   515 521 1
515 510 1   515 503 1   514 516 1   514 514 1   514 512 1   514 500 1   513 521 1   513 514 2
513 506 1   513 500 1   513 493 1   512 508 1   512 507 1   512 502 1   512 501 1   511 518 1
511 517 1   511 514 1   511 507 1   511 502 1   511 501 1   511 499 1   511 494 1   510 517 1
510 513 1   510 512 1   510 507 1   510 501 1   509 520 1   509 514 1   509 512 1   509 509 1
509 498 1   509 484 1   508 514 1   508 491 1   507 514 1   507 507 1   507 505 1   507 503 1
507 500 1   506 516 1   506 512 1   506 500 1   506 491 1   505 530 1   505 523 1   505 502 1
505 498 1   504 509 1   504 504 1   504 502 1   504 500 2   503 512 1   503 509 1   503 508 2
502 514 2   502 513 2   502 508 1   502 505 1   502 500 1   502 495 1   501 509 1   501 506 1
501 502 1   501 501 1   501 500 3   501 497 2   501 495 1   500 517 1   500 516 1   500 513 1
500 501 2   500 500 171   500 492 1   499 546 1   499 501 1   499 500 1   498 507 1   498 503 1
498 500 2   497 518 1   497 514 1   497 506 1   497 495 1   497 489 1   497 483 1   496 505 1
496 493 1   496 487 1   495 516 1   495 495 1   494 498 1   494 481 1   493 504 1   492 503 1
492 501 1   492 492 1   491 516 1   491 503 1   491 486 1   490 505 1   490 492 1   489 498 1
487 500 1   483 481 1   482 495 1   480 495 1   479 495 1   473 489 1   468 499 1
"""
score_pairs = np.fromstring(score_pair_table, sep=" ", dtype=int).reshape(-1, 3)
validation_units, holdout_units, frequencies = score_pairs.T
assert np.all((score_pairs[:, :2] >= 0) & (score_pairs[:, :2] <= 1000))
assert np.all(frequencies > 0) and frequencies.sum() == 6425

In [2]:
focus_score = 958  # A displayed macro AUC of 0.958.
focus_mask = validation_units == focus_score
focus_total = int(frequencies[focus_mask].sum())
focus_pairs = score_pairs[focus_mask]
mode_row = focus_pairs[focus_pairs[:, 2].argmax()]

# Restore the multiplicities before computing record-level medians.
differences = validation_units - holdout_units
median_differences = {}
for label, mask in [("all", np.ones_like(focus_mask)), ("at_0_958", focus_mask), ("other", ~focus_mask)]:
    values = np.repeat(differences[mask], frequencies[mask])
    median_differences[label] = float(np.median(values) / 1000)

observed_summary = {
    "records": int(frequencies.sum()),
    "validation_score": focus_score / 1000,
    "records_at_validation_score": focus_total,
    "distinct_holdout_scores": len(focus_pairs),
    "holdout_range": [float(focus_pairs[:, 1].min() / 1000), float(focus_pairs[:, 1].max() / 1000)],
    "most_frequent_holdout_score": float(mode_row[1] / 1000),
    "records_at_most_frequent_holdout_score": int(mode_row[2]),
    "median_validation_minus_holdout": median_differences,
}
print(f"Matched records                           {observed_summary['records']:,}")
print(f"Records with validation AUC {focus_score / 1000:.3f}         {focus_total:,}")
print("\nHoldout AUC   Records    Share among validation ties")
for _, score, count in focus_pairs:
    print(f"{score / 1000:>11.3f}   {count:>7,d}    {count / focus_total:>6.1%}")
print("\nMedian validation minus holdout AUC")
for group, difference in median_differences.items():
    print(f"{group:>12}   {difference:.3f}")

Matched records                           6,425
Records with validation AUC 0.958         1,638

Holdout AUC   Records    Share among validation ties
      0.946         3      0.2%
      0.945        12      0.7%
      0.944        53      3.2%
      0.943       272     16.6%
      0.942     1,239     75.6%
      0.941        24      1.5%
      0.940         7      0.4%
      0.939         3      0.2%
      0.938         2      0.1%
      0.937         2      0.1%
      0.936         1      0.1%
      0.935         1      0.1%
      0.934         1      0.1%
      0.933         2      0.1%
      0.931         1      0.1%
      0.930         1      0.1%
      0.927         4      0.2%
      0.926         1      0.1%
      0.925         1      0.1%
      0.923         3      0.2%
      0.922         1      0.1%
      0.920         1      0.1%
      0.919         1      0.1%
      0.909         1      0.1%
      0.899         1      0.1%

Median validation minus holdout AUC
         all 

Of the 1,638 records with validation AUC 0.958, 1,239 (75.6%) had holdout AUC 0.942. Across the group, the holdout score took 25 distinct values between 0.899 and 0.946. The median difference between validation and holdout was 0.016 both inside and outside the group. These summaries show the spread behind a single displayed number. Attributing the differences to a model change would require the configurations and their predictions.

## Simulating model variants

The historical summaries omit the configurations behind each score. In the simulation we know exactly which component changed, so we can test how often a small validation improvement also appears on holdout. Each trial generates 1,300 synthetic studies and splits them into 390 validation and 910 holdout studies. All variants see the same studies. Twelve independent binary findings have prevalences from 15% to 60%. The shared baseline has equal-variance Gaussian prediction scores within each class, with a target mean AUC of 0.958 across findings.

Three optional prediction components produce eight model configurations. Each configuration has four runs with independent prediction noise. A component’s signal strength is redrawn for each trial, then held fixed across studies and configurations. A calibration step maps signal strength to the approximate change in macro AUC when adding that component alone to the baseline.

In [3]:
from statistics import NormalDist

import numpy as np
from scipy.stats import rankdata

N_STUDIES = 1300
N_VALIDATION = 390
N_FINDINGS = 12
PREVALENCE = np.linspace(0.15, 0.60, N_FINDINGS)
BASELINE_AUC = np.linspace(0.925, 0.991, N_FINDINGS)
SEPARATION = np.sqrt(2) * np.array([NormalDist().inv_cdf(a) for a in BASELINE_AUC])
N_COMPONENTS = 3
COMPONENT_WEIGHT = 0.10
EFFECT_SDS = (0.0005, 0.001, 0.002)
PRIMARY_SD = 0.001
RUN_NOISE = 0.01
N_RUNS = 4
N_TRIALS = 600
SCORE_STEP = 0.001
CALIBRATION_SEED = 7
SIMULATION_SEED = 20261022

# Each bit identifies an optional component, so flipping one bit changes one component.
CONFIGURATIONS = np.array([
    [(config >> bit) & 1 for bit in range(N_COMPONENTS)]
    for config in range(2 ** N_COMPONENTS)
], dtype=np.float32)
N_CONFIGS = len(CONFIGURATIONS)


def macro_auc(scores, labels):
    """Compute per-finding AUC from score ranks, then average over findings."""
    n = scores.shape[-1]
    positives = labels.sum(axis=-1)
    if np.any((positives == 0) | (positives == n)):
        raise ValueError("AUC requires both positive and negative studies for every finding in each split.")
    ranks = rankdata(scores, axis=-1, method="average")
    rank_sum = (ranks * labels).sum(axis=-1)
    return ((rank_sum - positives * (positives + 1) / 2) / (positives * (n - positives))).mean(axis=-1)


def draw_studies(n, rng):
    labels = rng.random((N_FINDINGS, n)) < PREVALENCE[:, None]
    baseline = (SEPARATION[:, None] * labels + rng.standard_normal((N_FINDINGS, n))).astype(np.float32)
    noise = rng.standard_normal((N_COMPONENTS, N_FINDINGS, n), dtype=np.float32)
    return labels, baseline, noise


def component_scores(strengths, labels, noise):
    scale = np.asarray(strengths, dtype=np.float32)[:, None, None] * SEPARATION.astype(np.float32)[None, :, None]
    return scale * labels[None] + noise


# Fit a local linear approximation using one component and a large synthetic sample.
cal_labels, cal_baseline, cal_noise = draw_studies(300_000, np.random.default_rng(CALIBRATION_SEED))
base_auc = macro_auc(cal_baseline, cal_labels)
strength_grid = np.array([-0.6, -0.3, 0.0, 0.3, 0.6])
effects = np.array([
    macro_auc(cal_baseline + COMPONENT_WEIGHT * component_scores([g], cal_labels, cal_noise[:1])[0], cal_labels) - base_auc
    for g in strength_grid
])
slope, intercept = np.polyfit(strength_grid, effects, 1)
fit_error = np.max(np.abs(effects - (intercept + slope * strength_grid)))
del cal_labels, cal_baseline, cal_noise
print(f"Calibration baseline AUC {base_auc:.4f} | maximum absolute fit error {fit_error:.2g}")

Calibration baseline AUC 0.9578 | maximum absolute fit error 0.00039


In [4]:
def simulate(effect_sd, seed):
    rng = np.random.default_rng(seed)
    validation = np.empty((N_TRIALS, N_CONFIGS, N_RUNS))
    holdout = np.empty_like(validation)
    for trial in range(N_TRIALS):
        labels, baseline, noise = draw_studies(N_STUDIES, rng)
        strengths = rng.normal(-intercept / slope, effect_sd / slope, N_COMPONENTS)
        components = component_scores(strengths, labels, noise)
        variants = baseline[None] + COMPONENT_WEIGHT * np.tensordot(CONFIGURATIONS, components, axes=1)
        runs = variants[:, None] + RUN_NOISE * rng.standard_normal(
            (N_CONFIGS, N_RUNS, N_FINDINGS, N_STUDIES), dtype=np.float32
        )
        indices = rng.permutation(N_STUDIES)
        val_idx = np.sort(indices[:N_VALIDATION])
        hold_idx = np.sort(indices[N_VALIDATION:])
        validation[trial] = macro_auc(runs[..., val_idx], labels[:, val_idx])
        holdout[trial] = macro_auc(runs[..., hold_idx], labels[:, hold_idx])
    return validation, holdout

### Measuring stability and comparing pairs

For each pair of configurations differing in one component, compare the direction of the AUC change across splits using the first run of each configuration. For changes no larger than 0.001 on validation, report the fraction with the same sign on holdout. Rank correlations describe the ordering of runs within a narrow validation window and among repeats of a single configuration. The AUC and correlation calculations assign average ranks to tied scores, including ties from finite-precision synthetic predictions. A group with a constant score on either split has no defined correlation and is omitted from the correlation average.

A second comparison uses the first two runs of each configuration as candidates. Select the highest-validation run as the reference, then pair it with its repeat, the next-highest validation run, or the strongest validation run among its one-component neighbors. All selections use full-precision validation scores. For each pair, measure how much its better holdout score exceeds the reference’s holdout score. This retrospective best-of-two metric measures coverage across two candidates. Deploying a single candidate would require a separate selection rule.

In [5]:
def rank_correlation(a, b):
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return None
    return np.corrcoef(rankdata(a, method="average"), rankdata(b, method="average"))[0, 1]


def summarize(validation, holdout):
    val_changes, hold_changes = [], []
    for bit in range(N_COMPONENTS):
        for config in np.flatnonzero(CONFIGURATIONS[:, bit] == 0):
            neighbor = config ^ (1 << bit)
            val_changes.append(validation[:, neighbor, 0] - validation[:, config, 0])
            hold_changes.append(holdout[:, neighbor, 0] - holdout[:, config, 0])
    val_changes, hold_changes = map(np.concatenate, (val_changes, hold_changes))
    close = np.abs(val_changes) <= SCORE_STEP
    agreement = np.sign(val_changes) == np.sign(hold_changes)
    repeat_sd = np.std(holdout[:, :, 0] - holdout[:, :, 1])

    window_corr, repeat_corr = [], []
    gains = np.empty((len(validation), 3))
    two_from_same_config = []
    for trial, (val, hold) in enumerate(zip(validation, holdout)):
        window = np.abs(val - np.median(val)) <= SCORE_STEP / 2
        if window.sum() >= 6:
            window_corr.append(rank_correlation(val[window], hold[window]))
        repeat_corr.extend(rank_correlation(v, h) for v, h in zip(val, hold))

        # The first two columns supply the candidate runs for every configuration.
        candidates = val[:, :2]
        order = np.argsort(-candidates.ravel())
        config, run = divmod(int(order[0]), 2)
        second_config, second_run = divmod(int(order[1]), 2)
        neighbors = [config ^ (1 << bit) for bit in range(N_COMPONENTS)]
        neighbor_idx, neighbor_run = np.unravel_index(np.argmax(candidates[neighbors]), (N_COMPONENTS, 2))
        partners = [hold[config, 1 - run], hold[second_config, second_run], hold[neighbors[neighbor_idx], neighbor_run]]
        reference = hold[config, run]
        gains[trial] = np.maximum(reference, partners) - reference
        two_from_same_config.append(config == second_config)

    window_corr = [v for v in window_corr if v is not None]
    repeat_corr = [v for v in repeat_corr if v is not None]
    difference = gains[:, 2] - gains[:, 0]
    tolerance = 3 * repeat_sd
    return {
        "validation_score_sd": validation[:, 0, 0].std(),
        "holdout_score_sd": holdout[:, 0, 0].std(),
        "validation_repeat_difference_sd": (validation[:, :, 0] - validation[:, :, 1]).std(),
        "holdout_repeat_difference_sd": repeat_sd,
        "validation_component_difference_sd": val_changes.std(),
        "holdout_component_difference_sd": hold_changes.std(),
        "sign_agreement": agreement.mean(), "small_change_share": close.mean(),
        "small_change_sign_agreement": agreement[close].mean(),
        "window_rank_correlation": np.mean(window_corr) if window_corr else None,
        "repeat_rank_correlation": np.mean(repeat_corr) if repeat_corr else None,
        "top_two_same_configuration_share": np.mean(two_from_same_config),
        "repeat_pair_gain": gains[:, 0].mean(), "top_two_pair_gain": gains[:, 1].mean(),
        "neighbor_pair_gain": gains[:, 2].mean(),
        "neighbor_better_share": np.mean(difference > tolerance),
        "neighbor_worse_share": np.mean(difference < -tolerance),
    }

In [6]:
simulation_results = {}
for offset, effect_sd in enumerate(EFFECT_SDS):
    split_scores = simulate(effect_sd, seed=SIMULATION_SEED + offset)
    simulation_results[effect_sd] = summarize(*split_scores)
    print(f"Completed {N_TRIALS} trials with component effect SD {effect_sd:g}")

print("Primary setting")
for name, value in simulation_results[PRIMARY_SD].items():
    print(f"{name.replace('_', ' '):37} {value:.6f}" if value is not None else f"{name} unavailable")

print("\nEffect SD | small-change agreement | repeat gain | top-two gain | neighbor gain")
for effect_sd, stats in simulation_results.items():
    print(f"{effect_sd:9.4f} | {stats['small_change_sign_agreement']:22.1%} | "
          f"{stats['repeat_pair_gain']:11.6f} | {stats['top_two_pair_gain']:12.6f} | {stats['neighbor_pair_gain']:13.6f}")

Completed 600 trials with component effect SD 0.0005
Completed 600 trials with component effect SD 0.001
Completed 600 trials with component effect SD 0.002
Primary setting
validation score sd                   0.002993
holdout score sd                      0.002056
validation repeat difference sd       0.000075
holdout repeat difference sd          0.000041
validation component difference sd    0.001058
holdout component difference sd       0.001006
sign agreement                        0.867778
small change share                    0.639167
small change sign agreement           0.795089
window rank correlation               0.393442
repeat rank correlation               0.011000
top two same configuration share      0.878333
repeat pair gain                      0.000016
top two pair gain                     0.000036
neighbor pair gain                    0.000082
neighbor better share                 0.166667
neighbor worse share                  0.001667

Effect SD | small-change ag

### Reading the simulation results

At the default effect size of 0.001, switching one component moved validation and holdout AUC in the same direction in **87%** of comparisons. Restricting the comparison to validation changes of at most 0.001 in magnitude reduced agreement to **80%**. A small improvement in this experiment was informative, but it could reverse on a different set of studies.

A single baseline run had a validation AUC standard deviation of 0.0030 across newly generated cohorts. On the same validation studies, the difference between two repeats of a configuration had a standard deviation of 0.000075. Changing the studies and repeating the run are two different sources of variation.

The sensitivity table shows how the results change with the assumed size of a component effect. The reported better and worse shares use a tolerance of three times the holdout repeat-difference standard deviation. The simulation assumes independent findings. Patterns of dependence in real scans could change how much the average AUC varies across cohorts.

## Save the results

Run this cell to write `evaluation_summary.json` in the notebook’s working directory. It records the observed score summary, simulation settings, and metrics for each component-effect size. Differences and gains are in absolute AUC units. Shares are fractions from 0 to 1.

In [7]:
import json
from pathlib import Path

summary = {
    "observed_scores": observed_summary,
    "simulation_settings": {
        "findings": N_FINDINGS,
        "synthetic_studies_per_trial": N_STUDIES,
        "validation_studies": N_VALIDATION,
        "holdout_studies": N_STUDIES - N_VALIDATION,
        "trials_per_setting": N_TRIALS,
        "baseline_auc_by_finding": BASELINE_AUC.tolist(),
        "prevalence_by_finding": PREVALENCE.tolist(),
        "optional_components": N_COMPONENTS,
        "component_weight": COMPONENT_WEIGHT,
        "runs_per_configuration": N_RUNS,
        "run_noise_sd": RUN_NOISE,
        "validation_gap_cutoff": SCORE_STEP,
        "primary_component_effect_sd": PRIMARY_SD,
        "calibration_seed": CALIBRATION_SEED,
        "simulation_seed_by_component_effect_sd": {str(sd): SIMULATION_SEED + i for i, sd in enumerate(EFFECT_SDS)},
    },
    "simulation_by_component_effect_sd": {str(sd): metrics for sd, metrics in simulation_results.items()},
}
output_path = Path("evaluation_summary.json")
output_path.write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Wrote {output_path.name}")

Wrote evaluation_summary.json
